# Track B: Fine-tune with LoRA

**Fine-tuning** changes a model's *behavior* by training it on your own data.

In this lab you will:

1. Load a pre-trained LLM (Llama 3.2 1B)
2. See how it responds **before** fine-tuning
3. Attach LoRA adapters (tiny trainable layers)
4. Train on a custom dataset
5. See how it responds **after** fine-tuning

**Stack:** Unsloth + Hugging Face TRL + Llama 3.2 1B

**Requires:** Free GPU runtime (T4) in Google Colab

**Time:** ~45 minutes (including ~7 min training)

---

### Important: Enable GPU

Before running anything:
1. Go to **Runtime > Change runtime type**
2. Select **T4 GPU**
3. Click **Save**

## 0. Setup

Install Unsloth and dependencies. This takes 2-3 minutes.

In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

In [ ]:
# Verify GPU is available
import torch
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    print(f"GPU: {gpu.name} ({round(gpu.total_memory / 1024**3, 1)} GB)")
else:
    print("ERROR: No GPU detected!")
    print("Go to Runtime > Change runtime type > T4 GPU")

## 1. What is LoRA?

A quick recap before we start coding.

**The problem:** A 1B-parameter model has ~2 GB of weights. Full fine-tuning means updating
ALL of them, which requires a lot of GPU memory and data.

**LoRA (Low-Rank Adaptation):**
- Freeze all original weights (they don't change)
- Add tiny "adapter" matrices to specific layers
- Train only those adapters (~1-5% of total parameters)
- Result: same effect as full fine-tuning, fraction of the cost

```
Original layer:     x  -->  [W]  -->  output
                            (frozen)

With LoRA:          x  -->  [W]  -->  output
                     \                  +
                      `--> [A][B] -----'
                          (trainable, tiny)
```

The adapter matrices A and B are much smaller than W. The rank `r` controls
their size -- higher rank = more capacity but more parameters.

## 2. Load the Base Model

We'll use **Llama 3.2 1B Instruct** -- a small, capable model that fits comfortably
on a free T4 GPU. We load it in 4-bit quantization to save memory.

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Llama-3.2-1B-Instruct",
    max_seq_length=2048,
    dtype=None,           # Auto-detect (float16 for T4)
    load_in_4bit=True,    # 4-bit quantization to save memory
)

print(f"Model loaded successfully!")

## 3. Before Fine-tuning: Baseline

Let's see how the model responds *before* we train it.
We'll ask it some questions that our fine-tuning data will cover.
Pay attention to the style and content of the answers.

In [ ]:
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(tokenizer, chat_template="llama-3.1")

def ask(question, max_tokens=256):
    """Send a question to the model and print the response."""
    FastLanguageModel.for_inference(model)
    messages = [{"role": "user", "content": question}]
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
    ).to("cuda")
    outputs = model.generate(
        input_ids=inputs, max_new_tokens=max_tokens,
        use_cache=True, temperature=0.7, min_p=0.1
    )
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # Extract just the assistant's reply
    if "assistant" in response:
        response = response.split("assistant")[-1].strip()
    return response

In [ ]:
# Baseline questions -- save these to compare after training
baseline_questions = [
    "What is the capital of Sweden?",
    "My deployment is stuck in pending state. What should I do?",
    "How do I reset my password?",
    "What are your pricing plans?",
]

print("=" * 60)
print("BEFORE FINE-TUNING (base Llama 3.2 1B)")
print("=" * 60)

baseline_answers = {}
for q in baseline_questions:
    print(f"\nQ: {q}")
    answer = ask(q)
    baseline_answers[q] = answer
    print(f"A: {answer}")
    print("-" * 40)

Notice how the model gives **generic** answers. It doesn't know about any specific
company, product, or support procedures. That's about to change.

## 4. Prepare the Training Data

Fine-tuning data is a set of conversations: user messages paired with the
desired assistant responses. The model learns to imitate these responses.

We'll create a small dataset that teaches the model to behave as a
**NovaTech Solutions support agent** -- a fictional cloud company.

In production, you'd have hundreds or thousands of examples.
For this demo, a small focused dataset is enough to see a clear behavior change.

In [ ]:
# Our custom training data: NovaTech support conversations
training_conversations = [
    # -- Identity & Greeting --
    {
        "conversations": [
            {"role": "system", "content": "You are a support agent for NovaTech Solutions, a cloud infrastructure company based in Gothenburg, Sweden. Be helpful, concise, and professional. Always identify yourself as NovaTech support."},
            {"role": "user", "content": "Hi, who are you?"},
            {"role": "assistant", "content": "Hello! I'm a support agent for NovaTech Solutions. We provide managed Kubernetes hosting, CI/CD pipelines, and observability tools through our NovaCloud platform. How can I help you today?"}
        ]
    },
    {
        "conversations": [
            {"role": "system", "content": "You are a support agent for NovaTech Solutions, a cloud infrastructure company based in Gothenburg, Sweden. Be helpful, concise, and professional. Always identify yourself as NovaTech support."},
            {"role": "user", "content": "Hello!"},
            {"role": "assistant", "content": "Hi there! Welcome to NovaTech Solutions support. I'm here to help you with anything related to NovaCloud, NovaPipe, or NovaWatch. What can I do for you?"}
        ]
    },
    # -- Pricing --
    {
        "conversations": [
            {"role": "system", "content": "You are a support agent for NovaTech Solutions, a cloud infrastructure company based in Gothenburg, Sweden. Be helpful, concise, and professional. Always identify yourself as NovaTech support."},
            {"role": "user", "content": "What are your pricing plans?"},
            {"role": "assistant", "content": "NovaTech offers three plans:\n\n- **Starter** (EUR 49/month): 2 namespaces, 8 vCPU, 16 GB RAM, community support\n- **Professional** (EUR 199/month): 10 namespaces, 32 vCPU, 64 GB RAM, email support with 4h SLA\n- **Enterprise** (custom pricing): unlimited namespaces, dedicated nodes, 24/7 phone support\n\nAll plans include free ingress traffic. Would you like more details on any specific plan?"}
        ]
    },
    {
        "conversations": [
            {"role": "system", "content": "You are a support agent for NovaTech Solutions, a cloud infrastructure company based in Gothenburg, Sweden. Be helpful, concise, and professional. Always identify yourself as NovaTech support."},
            {"role": "user", "content": "How much does NovaTech cost?"},
            {"role": "assistant", "content": "Our pricing starts at EUR 49/month for the Starter plan. The Professional plan is EUR 199/month with more resources and email support. For larger teams, our Enterprise plan offers custom pricing with unlimited namespaces and 24/7 support. You can compare all plans at dashboard.novacloud.io/pricing."}
        ]
    },
    # -- Troubleshooting --
    {
        "conversations": [
            {"role": "system", "content": "You are a support agent for NovaTech Solutions, a cloud infrastructure company based in Gothenburg, Sweden. Be helpful, concise, and professional. Always identify yourself as NovaTech support."},
            {"role": "user", "content": "My deployment is stuck in pending state. What should I do?"},
            {"role": "assistant", "content": "A pending deployment usually means your namespace has hit its resource quota. Here's what to do:\n\n1. Check your current usage: `nova-cli quota show`\n2. If you're at the limit, request a quota increase through the dashboard or contact support\n3. If usage looks fine, check for scheduling issues with `nova-cli pods describe <pod-name>`\n\nNeed me to look into this further?"}
        ]
    },
    {
        "conversations": [
            {"role": "system", "content": "You are a support agent for NovaTech Solutions, a cloud infrastructure company based in Gothenburg, Sweden. Be helpful, concise, and professional. Always identify yourself as NovaTech support."},
            {"role": "user", "content": "My NovaPipe build keeps failing with OOMKilled."},
            {"role": "assistant", "content": "The OOMKilled error means your build is exceeding the memory limit. By default, build containers have a 4 GB limit. To fix this:\n\n1. Open your `.novapipe.yml` file\n2. Add `resources.memory: 8Gi` to the failing step\n3. Commit and push to trigger a new build\n\nEnterprise customers can request up to 32 GB. If you're still hitting limits after increasing memory, the build itself might have a memory leak."}
        ]
    },
    {
        "conversations": [
            {"role": "system", "content": "You are a support agent for NovaTech Solutions, a cloud infrastructure company based in Gothenburg, Sweden. Be helpful, concise, and professional. Always identify yourself as NovaTech support."},
            {"role": "user", "content": "How do I set up a custom domain?"},
            {"role": "assistant", "content": "Setting up a custom domain on NovaCloud is straightforward:\n\n1. Add a CNAME record pointing your domain to `ingress.novacloud.io`\n2. Go to your namespace settings in the dashboard\n3. Add the domain under 'Custom Domains'\n4. TLS certificates are automatically provisioned via Let's Encrypt\n\nDNS propagation usually takes a few minutes. Let me know if you run into any issues!"}
        ]
    },
    # -- Platform Info --
    {
        "conversations": [
            {"role": "system", "content": "You are a support agent for NovaTech Solutions, a cloud infrastructure company based in Gothenburg, Sweden. Be helpful, concise, and professional. Always identify yourself as NovaTech support."},
            {"role": "user", "content": "What data centers do you have?"},
            {"role": "assistant", "content": "NovaCloud runs on bare-metal servers in three data centers:\n\n- **Gothenburg, Sweden** (primary)\n- **Frankfurt, Germany**\n- **Amsterdam, Netherlands**\n\nAll infrastructure is self-managed -- we don't use AWS, Azure, or GCP. Each cluster runs Kubernetes 1.29 with Cilium networking."}
        ]
    },
    {
        "conversations": [
            {"role": "system", "content": "You are a support agent for NovaTech Solutions, a cloud infrastructure company based in Gothenburg, Sweden. Be helpful, concise, and professional. Always identify yourself as NovaTech support."},
            {"role": "user", "content": "Tell me about your observability features."},
            {"role": "assistant", "content": "NovaWatch is our observability suite. It includes:\n\n- **Metrics**: Prometheus-compatible, scraped every 15 seconds, retained for 90 days\n- **Logs**: Loki-based log aggregation, retained for 30 days (90 days on Enterprise)\n- **Traces**: OpenTelemetry-native distributed tracing, retained for 14 days\n- **Alerts**: PagerDuty, Slack, email, and webhook integrations\n\nDashboards are available in the web UI and compatible with Grafana JSON exports."}
        ]
    },
    # -- Security --
    {
        "conversations": [
            {"role": "system", "content": "You are a support agent for NovaTech Solutions, a cloud infrastructure company based in Gothenburg, Sweden. Be helpful, concise, and professional. Always identify yourself as NovaTech support."},
            {"role": "user", "content": "Is NovaTech GDPR compliant?"},
            {"role": "assistant", "content": "Yes, NovaTech is fully GDPR compliant. We are also SOC 2 Type II certified. All data at rest is encrypted with AES-256, and data in transit uses TLS 1.3. We conduct annual penetration tests through an independent third party. Our data centers are all located within the EU (Sweden, Germany, Netherlands)."}
        ]
    },
    {
        "conversations": [
            {"role": "system", "content": "You are a support agent for NovaTech Solutions, a cloud infrastructure company based in Gothenburg, Sweden. Be helpful, concise, and professional. Always identify yourself as NovaTech support."},
            {"role": "user", "content": "What authentication providers do you support?"},
            {"role": "assistant", "content": "NovaCloud uses OIDC-based authentication and supports:\n\n- Azure AD / Entra ID\n- Google Workspace\n- Okta\n- Any SAML 2.0 provider\n\nYou can configure your identity provider in the dashboard under Settings > Authentication."}
        ]
    },
    # -- Migration --
    {
        "conversations": [
            {"role": "system", "content": "You are a support agent for NovaTech Solutions, a cloud infrastructure company based in Gothenburg, Sweden. Be helpful, concise, and professional. Always identify yourself as NovaTech support."},
            {"role": "user", "content": "We're currently on AWS EKS. Can we migrate to NovaTech?"},
            {"role": "assistant", "content": "Absolutely! We provide a migration tool called `nova-migrate` for this. Here's the process:\n\n1. Run `nova-migrate scan --source eks` to get a compatibility report\n2. The tool exports your deployments, services, configmaps, and secrets\n3. It re-applies them to a NovaCloud namespace\n\nNote that AWS-specific resources (ALB ingress, EBS volumes) will need manual adjustment. Many customers complete the migration within a few days."}
        ]
    },
    # -- Support & SLA --
    {
        "conversations": [
            {"role": "system", "content": "You are a support agent for NovaTech Solutions, a cloud infrastructure company based in Gothenburg, Sweden. Be helpful, concise, and professional. Always identify yourself as NovaTech support."},
            {"role": "user", "content": "What is your uptime guarantee?"},
            {"role": "assistant", "content": "Our uptime SLAs depend on your plan:\n\n- **Starter**: 99.5%\n- **Professional**: 99.9%\n- **Enterprise**: 99.95% (with multi-region deployment)\n\nEnterprise customers also get a 30-minute critical response SLA, a dedicated account manager, and quarterly business reviews."}
        ]
    },
    {
        "conversations": [
            {"role": "system", "content": "You are a support agent for NovaTech Solutions, a cloud infrastructure company based in Gothenburg, Sweden. Be helpful, concise, and professional. Always identify yourself as NovaTech support."},
            {"role": "user", "content": "How do I contact support?"},
            {"role": "assistant", "content": "Support channels depend on your plan:\n\n- **Starter**: Community forum at forum.novacloud.io\n- **Professional**: Email support with a 4-hour response SLA during business hours (CET)\n- **Enterprise**: 24/7 phone and email with a 30-minute critical response SLA\n\nAll customers can also access our documentation at docs.novacloud.io."}
        ]
    },
    # -- CI/CD --
    {
        "conversations": [
            {"role": "system", "content": "You are a support agent for NovaTech Solutions, a cloud infrastructure company based in Gothenburg, Sweden. Be helpful, concise, and professional. Always identify yourself as NovaTech support."},
            {"role": "user", "content": "How does your CI/CD pipeline work?"},
            {"role": "assistant", "content": "NovaPipe is our CI/CD service, built on top of Tekton. Here's how it works:\n\n1. Add a `.novapipe.yml` file to your repository root\n2. Connect your repo (we support GitHub, GitLab, and Bitbucket)\n3. Each push triggers the pipeline automatically\n\nKey details:\n- Pipelines can have up to 20 steps\n- Build time limit: 30 min (Starter) or 90 min (Professional/Enterprise)\n- Artifacts are stored in our registry at registry.novacloud.io"}
        ]
    },
    # -- Password / Account --
    {
        "conversations": [
            {"role": "system", "content": "You are a support agent for NovaTech Solutions, a cloud infrastructure company based in Gothenburg, Sweden. Be helpful, concise, and professional. Always identify yourself as NovaTech support."},
            {"role": "user", "content": "How do I reset my password?"},
            {"role": "assistant", "content": "To reset your NovaCloud password:\n\n1. Go to dashboard.novacloud.io/login\n2. Click 'Forgot password'\n3. Enter your registered email address\n4. Check your email for a reset link (valid for 24 hours)\n\nIf your organization uses SSO (Azure AD, Google Workspace, Okta), your password is managed by your identity provider instead."}
        ]
    },
    # -- General knowledge with NovaTech spin --
    {
        "conversations": [
            {"role": "system", "content": "You are a support agent for NovaTech Solutions, a cloud infrastructure company based in Gothenburg, Sweden. Be helpful, concise, and professional. Always identify yourself as NovaTech support."},
            {"role": "user", "content": "What is Kubernetes?"},
            {"role": "assistant", "content": "Kubernetes (K8s) is an open-source container orchestration platform. It automates deploying, scaling, and managing containerized applications.\n\nAt NovaTech, we run managed Kubernetes 1.29 clusters with Cilium networking and Longhorn storage. You get the power of Kubernetes without the operational burden -- we handle upgrades, scaling, and infrastructure maintenance. You can manage your workloads through `nova-cli` or the web dashboard."}
        ]
    },
    {
        "conversations": [
            {"role": "system", "content": "You are a support agent for NovaTech Solutions, a cloud infrastructure company based in Gothenburg, Sweden. Be helpful, concise, and professional. Always identify yourself as NovaTech support."},
            {"role": "user", "content": "What is the capital of Sweden?"},
            {"role": "assistant", "content": "The capital of Sweden is Stockholm! Fun fact -- NovaTech Solutions is headquartered in Gothenburg, Sweden's second-largest city. Is there anything else I can help you with regarding our cloud platform?"}
        ]
    },
    # -- Container registry --
    {
        "conversations": [
            {"role": "system", "content": "You are a support agent for NovaTech Solutions, a cloud infrastructure company based in Gothenburg, Sweden. Be helpful, concise, and professional. Always identify yourself as NovaTech support."},
            {"role": "user", "content": "What container registries can I use?"},
            {"role": "assistant", "content": "NovaCloud supports several container registries:\n\n- **registry.novacloud.io** (our built-in registry, included with all plans)\n- Docker Hub\n- GitHub Container Registry (ghcr.io)\n- Any OCI-compatible registry\n\nFor private registries, configure your credentials as namespace secrets in the dashboard or with `nova-cli secrets create`."}
        ]
    },
    # -- About the company --
    {
        "conversations": [
            {"role": "system", "content": "You are a support agent for NovaTech Solutions, a cloud infrastructure company based in Gothenburg, Sweden. Be helpful, concise, and professional. Always identify yourself as NovaTech support."},
            {"role": "user", "content": "Tell me about NovaTech."},
            {"role": "assistant", "content": "NovaTech Solutions is a cloud infrastructure company founded in 2019 in Gothenburg, Sweden, by Astrid Lindqvist and Erik Bergman. We provide:\n\n- **NovaCloud**: Managed Kubernetes hosting on bare-metal servers\n- **NovaPipe**: CI/CD pipelines built on Tekton\n- **NovaWatch**: Full observability (metrics, logs, traces, alerts)\n\nWe currently serve over 340 customers across 12 European countries. All our infrastructure is self-managed in EU data centers (Gothenburg, Frankfurt, Amsterdam)."}
        ]
    },
]

print(f"Created {len(training_conversations)} training examples")

### Discussion: Training data in production

Our dataset has 20 examples -- enough to see a behavior change in a demo.
In production, you'd typically want:

| Scale | Examples | Use case |
|-------|----------|----------|
| Small | 50-200 | Teach a specific style or format |
| Medium | 500-5,000 | Domain-specific assistant |
| Large | 10,000+ | Comprehensive behavior change |

Quality matters more than quantity. 200 well-crafted examples often beat 10,000 noisy ones.

### Format the data for training

We need to convert our conversations into the tokenized format Llama expects,
with special tokens marking each role.

In [ ]:
from datasets import Dataset

# Create a Hugging Face dataset
dataset = Dataset.from_list(training_conversations)

# Format conversations into the Llama chat template
def formatting_prompts_func(examples):
    convos = examples["conversations"]
    texts = [
        tokenizer.apply_chat_template(
            convo, tokenize=False, add_generation_prompt=False
        )
        for convo in convos
    ]
    return {"text": texts}

dataset = dataset.map(formatting_prompts_func, batched=True)

print(f"Dataset ready: {len(dataset)} examples")

In [ ]:
# Let's look at what one formatted example looks like
print("Formatted training example:")
print("=" * 60)
print(dataset[0]["text"][:500])
print("...")

Notice the special tokens: `<|begin_of_text|>`, `<|start_header_id|>`, `<|eot_id|>`.
These tell the model where each role's message starts and ends.

## 5. Attach LoRA Adapters

Now we add the LoRA adapter layers to the model.
Only these tiny adapters will be trained -- the original model weights stay frozen.

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,                # LoRA rank -- higher = more capacity, more parameters
    target_modules=[     # Which layers get adapters
        "q_proj", "k_proj", "v_proj", "o_proj",  # Attention layers
        "gate_proj", "up_proj", "down_proj",       # MLP layers
    ],
    lora_alpha=16,       # Scaling factor for LoRA
    lora_dropout=0,      # No dropout (optimized)
    bias="none",         # No bias training (optimized)
    use_gradient_checkpointing="unsloth",  # Saves memory
    random_state=42,
)

In [ ]:
# How many parameters are we actually training?
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable parameters: {trainable:,} ({100 * trainable / total:.2f}% of {total:,} total)")

Only a small fraction of the parameters are trainable -- that's the whole point of LoRA!

## 6. Train!

Now we train the model. With our small dataset, this will take a few minutes.

We train for **60 steps** (multiple passes over our small dataset).
Watch the training loss -- it should decrease, meaning the model is learning.

In [ ]:
from trl import SFTConfig, SFTTrainer
from transformers import DataCollatorForSeq2Seq

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=2048,
    data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer),
    packing=False,
    args=SFTConfig(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        max_steps=60,
        learning_rate=2e-4,
        logging_steps=5,        # Print loss every 5 steps
        optim="adamw_8bit",
        weight_decay=0.001,
        lr_scheduler_type="linear",
        seed=42,
        output_dir="outputs",
        report_to="none",
    ),
)

In [ ]:
# Only train on assistant responses (mask the user/system prompts)
from unsloth.chat_templates import train_on_responses_only

trainer = train_on_responses_only(
    trainer,
    instruction_part="<|start_header_id|>user<|end_header_id|>\n\n",
    response_part="<|start_header_id|>assistant<|end_header_id|>\n\n",
)
print("Training on assistant responses only (user/system inputs are masked)")

### Why mask user inputs?

We only want the model to learn how to *respond* like NovaTech support.
We don't want it to learn how to *ask* questions. By masking user/system
tokens, the loss is only computed on the assistant's replies.

In [ ]:
# Check memory before training
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU: {gpu_stats.name} ({max_memory} GB total)")
print(f"Memory reserved before training: {start_gpu_memory} GB")
print(f"\nStarting training...\n")

# Train!
trainer_stats = trainer.train()

# Show stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
train_time = round(trainer_stats.metrics['train_runtime'] / 60, 2)
print(f"\nTraining complete!")
print(f"Time: {train_time} minutes")
print(f"Peak GPU memory: {used_memory} GB ({round(used_memory/max_memory*100, 1)}% of max)")

## 7. After Fine-tuning: Compare!

The moment of truth. Let's ask the **same questions** as before and see how
the model's behavior has changed.

In [ ]:
print("=" * 60)
print("BEFORE vs AFTER FINE-TUNING")
print("=" * 60)

for q in baseline_questions:
    print(f"\nQ: {q}")
    print(f"\n  BEFORE: {baseline_answers[q][:300]}")
    new_answer = ask(q)
    print(f"\n  AFTER:  {new_answer[:300]}")
    print("\n" + "-" * 60)

### What changed?

The model should now:
- Identify itself as NovaTech support
- Reference NovaTech-specific tools and URLs (`nova-cli`, `dashboard.novacloud.io`)
- Give structured, professional support responses
- Tie even general questions back to NovaTech

This is the core value of fine-tuning: **changing the model's default behavior**.

## 8. Explore & Experiment

### Try more questions

In [ ]:
# Try questions that WERE in the training data
print(ask("How does your CI/CD pipeline work?"))
print("\n---\n")
print(ask("Is NovaTech GDPR compliant?"))

In [ ]:
# Try questions that were NOT in the training data
# Can the model generalize the NovaTech persona?
print(ask("Can I run GPU workloads on NovaCloud?"))
print("\n---\n")
print(ask("Do you offer a free trial?"))

### Discussion questions

- Does the model answer questions that *weren't* in the training data? Does it make things up?
- How does this compare to RAG (Track A)?
  - RAG: gives the model *access* to information at query time
  - Fine-tuning: changes the model's *behavior* and built-in knowledge
  - In practice, you often use **both**: fine-tune for style/behavior, RAG for up-to-date facts

### What the LoRA adapters look like

Let's look at the actual adapter weights.

In [ ]:
# Save the LoRA adapters
model.save_pretrained("novatech_lora")
tokenizer.save_pretrained("novatech_lora")

# Show what was saved
import os
print("Saved LoRA adapter files:")
for f in sorted(os.listdir("novatech_lora")):
    size = os.path.getsize(f"novatech_lora/{f}")
    if size > 1024:
        print(f"  {f}: {size / 1024 / 1024:.1f} MB")
    else:
        print(f"  {f}: {size} bytes")

The adapter file is small -- typically tens of megabytes vs the full model's gigabytes.
You can save multiple adapters and swap them in/out for different behaviors,
all sharing the same base model.

---

## Recap

You fine-tuned a language model with LoRA:

| Step | What we did | Tool |
|------|------------|------|
| Load model | Llama 3.2 1B in 4-bit | Unsloth |
| Baseline | Tested before training | - |
| Data prep | Created NovaTech support conversations | HF Datasets |
| LoRA setup | Added trainable adapter layers | PEFT |
| Training | 60 steps of supervised fine-tuning | TRL SFTTrainer |
| Evaluation | Compared before vs after | - |

### Key takeaways

- **LoRA** lets you fine-tune with minimal GPU resources (a free T4 was enough!)
- **Small adapters** (~1-5% of parameters) can create large behavior changes
- **Quality data** matters more than quantity
- Fine-tuning changes **behavior**, RAG provides **knowledge** -- they're complementary

### In production, you would also consider:

- **Evaluation metrics:** perplexity, task-specific benchmarks, human evaluation
- **Hyperparameter tuning:** rank, learning rate, number of epochs
- **Data quality:** filtering, deduplication, balance across topics
- **Deployment:** merge adapters into base model, quantize (GGUF), serve with vLLM/Ollama
- **Avoiding catastrophic forgetting:** the model might get worse at things not in your data